# Preparación de Datos — Terminal de Transporte

**Fase:** Modelo predictivo — preparación de datos (continuación de `01-exploracion-datos.ipynb`)

Este notebook parte de la muestra cruda ya descargada y guardada (`fase-1/data/raw/muestra_cruda.csv`, ~180.000 filas) y aplica las decisiones de limpieza respaldadas por la evidencia recogida en el notebook de exploración. No se descarga nada de nuevo desde la API.

**Regla que se mantiene:** cada eliminación o transformación de columnas se justifica citando la evidencia ya encontrada — nada se quita "porque sí".

**Salida de este notebook:** un dataset limpio, listo para modelar, guardado en `fase-1/data/processed/dataset_limpio.csv`.


## Configuración e imports

In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RAW_SAMPLE_PATH = "../data/raw/muestra_cruda.csv"
PROCESSED_PATH = "../data/processed/dataset_limpio.csv"


## 1. Cargar la muestra y reconstruir tipos y variables derivadas

Se repite exactamente el mismo reconocimiento de tipos y las mismas fórmulas del notebook de exploración (incluida la corrección de `duracion_viaje_horas`/`hora_salida_decimal` por el problema de medianoche en `fecha_salida`/`fecha_llegada`), para no depender de tener el otro notebook abierto.


In [3]:
df = pd.read_csv(RAW_SAMPLE_PATH)
print("Dimensiones iniciales:", df.shape)

date_cols = ["fecha_salida", "fecha_hora_salida_origen", "fecha_llegada", "hora_de_llegada"]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

df["pasajeros"] = pd.to_numeric(df["pasajeros"], errors="coerce")

categorical_cols = ["estado", "nombre_sucursal", "empresa", "ruta_origen",
                    "ruta_destino", "subregi_n", "clase_veh_culo"]
for c in categorical_cols:
    df[c] = df[c].astype("category")

# Formula corregida (ver 01-exploracion-datos.ipynb, Paso 6): fecha_salida/fecha_llegada
# estan truncadas a medianoche en ~65.7% de las filas; se usa fecha_hora_salida_origen
# con respaldo en fecha_salida, y hora_de_llegada como llegada real.
salida_real = df["fecha_hora_salida_origen"].fillna(df["fecha_salida"])
llegada_real = df["hora_de_llegada"]

df["duracion_viaje_horas"] = (llegada_real - salida_real).dt.total_seconds() / 3600
df["hora_salida_decimal"] = salida_real.dt.hour + salida_real.dt.minute / 60
df["dia_semana"] = df["fecha_salida"].dt.dayofweek  # 0 = lunes

df[["pasajeros", "duracion_viaje_horas", "hora_salida_decimal", "dia_semana"]].describe()


Dimensiones iniciales: (180000, 12)


,pasajeros,duracion_viaje_horas,hora_salida_decimal,dia_semana
count,180000.000000,180000.000000,180000.000000,180000.000000
mean,11.464039,3.545915,12.295768,2.962800
std,10.765276,4.936227,4.794846,1.978786
min,0.000000,-585.900000,0.000000,0.000000
25%,4.000000,1.266667,8.250000,1.000000
50%,8.000000,1.983333,12.333333,3.000000
75%,15.000000,4.066667,16.000000,5.000000
max,60.000000,757.550000,23.983333,6.000000


## 2. Eliminar columnas de varianza cero

**`estado`** y **`ruta_destino`** se confirmaron constantes en la exploración (Paso 4 de `01-exploracion-datos.ipynb`): `estado` solo tiene "LLEGADA"/"Llegada" (inconsistencia de mayúsculas, mismo valor), y `ruta_destino` es "MEDELLIN" en el 100% de las filas. Ninguna de las dos puede discriminar nada para el modelo — se eliminan.


In [4]:
print("Valores unicos de estado antes de eliminar:", df["estado"].unique().tolist())
print("Valores unicos de ruta_destino antes de eliminar:", df["ruta_destino"].unique().tolist())

df = df.drop(columns=["estado", "ruta_destino"])
print("\nColumnas restantes:", list(df.columns))


Valores unicos de estado antes de eliminar: ['Llegada', 'LLEGADA']
Valores unicos de ruta_destino antes de eliminar: ['MEDELLIN']

Columnas restantes: ['nombre_sucursal', 'empresa', 'fecha_salida', 'fecha_hora_salida_origen', 'fecha_llegada', 'hora_de_llegada', 'ruta_origen', 'subregi_n', 'clase_veh_culo', 'pasajeros', 'duracion_viaje_horas', 'hora_salida_decimal', 'dia_semana']


## 3. Normalizar inconsistencias de mayúsculas en categóricas

La exploración encontró que `nombre_sucursal` (4 valores en vez de 2 reales) y `subregi_n` (10 valores en vez de 5 reales) están duplicadas por mayúsculas/minúsculas. Se normaliza a mayúsculas en todas las categóricas restantes por seguridad, aunque algunas (`clase_veh_culo`, `empresa`, `ruta_origen`) ya vinieron consistentes en esta muestra.


In [5]:
categorical_cols_final = ["nombre_sucursal", "empresa", "ruta_origen", "subregi_n", "clase_veh_culo"]

print("Cardinalidad ANTES de normalizar:")
for c in categorical_cols_final:
    print(f"  {c}: {df[c].nunique()}")

for c in categorical_cols_final:
    df[c] = df[c].astype(str).str.upper().str.strip().astype("category")

print("\nCardinalidad DESPUES de normalizar:")
for c in categorical_cols_final:
    print(f"  {c}: {df[c].nunique()}")


Cardinalidad ANTES de normalizar:
  nombre_sucursal: 4
  empresa: 137
  ruta_origen: 375
  subregi_n: 10
  clase_veh_culo: 5

Cardinalidad DESPUES de normalizar:
  nombre_sucursal: 2
  empresa: 137
  ruta_origen: 375
  subregi_n: 5
  clase_veh_culo: 5


In [6]:
print(df["nombre_sucursal"].value_counts())
print()
print(df["subregi_n"].value_counts())


nombre_sucursal
TERMINAL DEL NORTE    132180
TERMINAL DEL SUR       47820
Name: count, dtype: int64

subregi_n
ORIENTE      69927
NORTE        53167
SUR          28678
OCCIDENTE    16584
CENTRO       11644
Name: count, dtype: int64


## 4. Filtrar filas con `duracion_viaje_horas` inválida

La exploración (Paso 7) encontró 18 filas negativas (0.010%, imposible: llegada antes que la salida) y un máximo de 757.6 horas (31 días) — errores de captura, no viajes reales. La regla estadística del IQR marcaba como "atípico" cualquier valor por encima de 8.27 horas, pero eso incluía viajes reales y repetidos como Montería → Medellín (8-11 horas), una ruta genuinamente larga.

**Decisión (provisional, a confirmar con el equipo/profesor):** eliminar las duraciones negativas y las mayores a 24 horas — ninguna de las rutas de este dataset debería tomar más de un día completo. Se documenta el corte exacto para poder ajustarlo si el equipo decide otro criterio.


In [7]:
n_antes = len(df)
n_negativas = (df["duracion_viaje_horas"] < 0).sum()
n_mayor_24h = (df["duracion_viaje_horas"] > 24).sum()

print(f"Filas totales antes del filtro: {n_antes:,}")
print(f"Duraciones negativas: {n_negativas:,} ({n_negativas/n_antes*100:.3f}%)")
print(f"Duraciones > 24 horas: {n_mayor_24h:,} ({n_mayor_24h/n_antes*100:.3f}%)")

df = df[(df["duracion_viaje_horas"] >= 0) & (df["duracion_viaje_horas"] <= 24)].copy()

n_despues = len(df)
print(f"\nFilas despues del filtro: {n_despues:,} (se eliminaron {n_antes - n_despues:,}, {(n_antes-n_despues)/n_antes*100:.3f}%)")
print(f"\nNueva distribucion de duracion_viaje_horas:")
print(df["duracion_viaje_horas"].describe())


Filas totales antes del filtro: 180,000
Duraciones negativas: 18 (0.010%)
Duraciones > 24 horas: 271 (0.151%)

Filas despues del filtro: 179,711 (se eliminaron 289, 0.161%)

Nueva distribucion de duracion_viaje_horas:
count    179711.000000
mean          3.495222
std           3.526202
min           0.000000
25%           1.266667
50%           1.983333
75%           4.050000
max          24.000000
Name: duracion_viaje_horas, dtype: float64


## 5. Eliminar las columnas de fecha originales

Ya se extrajo de ellas todo lo necesario (`duracion_viaje_horas`, `hora_salida_decimal`, `dia_semana`). Se eliminan las 4, cada una por una razón distinta:

| Columna | Por qué se elimina |
|---|---|
| `fecha_llegada` | Fuga de información — es una de las dos columnas usadas para calcular el objetivo |
| `hora_de_llegada` | Fuga de información — es la que se usó como llegada real para calcular `duracion_viaje_horas` |
| `fecha_hora_salida_origen` | No es fuga (se conoce antes de llegar), pero su información útil ya está en `hora_salida_decimal`; mantenerla aparte solo reintroduce su ~21-33% de nulos sin aportar nada nuevo |
| `fecha_salida` | No es fuga, pero ya se extrajo todo lo útil (`dia_semana`, y vía `salida_real`, `hora_salida_decimal`) |


In [8]:
df = df.drop(columns=["fecha_salida", "fecha_hora_salida_origen", "fecha_llegada", "hora_de_llegada"])
print("Columnas restantes:", list(df.columns))


Columnas restantes: ['nombre_sucursal', 'empresa', 'ruta_origen', 'subregi_n', 'clase_veh_culo', 'pasajeros', 'duracion_viaje_horas', 'hora_salida_decimal', 'dia_semana']


## 6. Categorías de alta cardinalidad — pendiente, sin resolver

`ruta_origen` y `empresa` tienen muchas categorías distintas. No se agrupan ni se tocan en este notebook — queda pendiente de decidir con el equipo/profesor qué estrategia usar (agrupar categorías raras en "OTROS", *frequency encoding*, u otra). Se deja documentada la situación para esa conversación.


In [9]:
for c in ["ruta_origen", "empresa"]:
    conteo = df[c].value_counts()
    print(f"=== {c}: {df[c].nunique()} categorias ===")
    print(f"Categorias con menos de 30 filas en la muestra: {(conteo < 30).sum()} de {len(conteo)}")
    print(f"Filas afectadas por esas categorias raras: {conteo[conteo < 30].sum():,} ({conteo[conteo < 30].sum()/len(df)*100:.2f}%)")
    print()


=== ruta_origen: 375 categorias ===
Categorias con menos de 30 filas en la muestra: 64 de 375
Filas afectadas por esas categorias raras: 706 (0.39%)

=== empresa: 137 categorias ===
Categorias con menos de 30 filas en la muestra: 47 de 137
Filas afectadas por esas categorias raras: 384 (0.21%)



## 7. Filas con valores faltantes restantes

Tras quitar `fecha_hora_salida_origen` (que concentraba casi todos los nulos relevantes), se revisa si queda algún nulo suelto en las columnas finales.


In [10]:
nulos_restantes = df.isnull().sum()
print(nulos_restantes[nulos_restantes > 0])


empresa    11
dtype: int64


In [11]:
n_antes = len(df)
df = df.dropna()
n_despues = len(df)
print(f"Filas eliminadas por nulos residuales: {n_antes - n_despues} ({(n_antes-n_despues)/n_antes*100:.4f}%)")
print(f"Filas finales: {n_despues:,}")


Filas eliminadas por nulos residuales: 11 (0.0061%)
Filas finales: 179,700


**Justificación:** los nulos que quedan (en `empresa`) son una fracción despreciable (menos de 0.01% de las filas) — a diferencia de `fecha_hora_salida_origen` (que sí tenía un porcentaje alto y una razón identificada, MAR por año), aquí no hay evidencia de un patrón, y eliminar estas pocas filas no distorsiona el dataset. No se justifica una estrategia de imputación para una fracción tan pequeña.


## 8. Dataset final

Columnas resultantes: variable objetivo + predictoras (categóricas y numéricas). `ruta_origen` y `empresa` se conservan tal cual (alta cardinalidad, decisión de codificación pendiente para la etapa de modelado — ver Sección 6).


In [12]:
print("Dimensiones finales:", df.shape)
print("\nColumnas y tipos:")
print(df.dtypes)
df.head()


Dimensiones finales: (179700, 9)

Columnas y tipos:
nombre_sucursal         category
empresa                 category
ruta_origen             category
subregi_n               category
clase_veh_culo          category
pasajeros                  int64
duracion_viaje_horas     float64
hora_salida_decimal      float64
dia_semana                 int32
dtype: object


,nombre_sucursal,empresa,ruta_origen,subregi_n,clase_veh_culo,pasajeros,duracion_viaje_horas,hora_salida_decimal,dia_semana
0,TERMINAL DEL NORTE,EXPRE - BELMIRA S.A.,SAN PEDRO DE LOS MILAGROS,NORTE,AUTOMOVIL,1,0.833333,5.00,2
1,TERMINAL DEL SUR,COTRACIBOL,BOLIVAR,SUR,AUTOMOVIL,1,2.166667,5.00,2
2,TERMINAL DEL SUR,FLOTA FREDONIA S.A.S,FREDONIA,SUR,AUTOMOVIL,1,1.683333,6.00,2
3,TERMINAL DEL NORTE,JUAN B VASQUEZ,GUADALUPE,NORTE,AUTOMOVIL,2,2.516667,5.25,2
4,TERMINAL DEL SUR,COTRACIBOL,BOLIVAR,SUR,AUTOMOVIL,1,2.966667,5.00,2


In [13]:
import os
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f"Dataset limpio guardado en: {PROCESSED_PATH}")
print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")


Dataset limpio guardado en: ../data/processed/dataset_limpio.csv
Filas: 179,700 | Columnas: 9


# Resumen de lo aplicado en este notebook

1. Se eliminaron `estado` y `ruta_destino` (varianza cero, confirmado en la exploración).
2. Se normalizaron mayúsculas en `nombre_sucursal`, `empresa`, `ruta_origen`, `subregi_n`, `clase_veh_culo` — confirmó que `nombre_sucursal` son 2 terminales reales y `subregi_n` son 5 subregiones reales.
3. Se filtraron filas de `duracion_viaje_horas` inválidas: negativas y mayores a 24 horas (decisión provisional, con la evidencia de Montería vs. Condoto como respaldo — a confirmar con el equipo/profesor).
4. Se eliminaron las 4 columnas de fecha originales (`fecha_salida`, `fecha_hora_salida_origen`, `fecha_llegada`, `hora_de_llegada`) — dos por fuga de información, dos porque ya se extrajo su información útil.
5. Se dejó señalada, sin resolver, la decisión de categorías raras en `ruta_origen`/`empresa` (pendiente de hablar con el equipo/profesor).
6. Se eliminaron las pocas filas con nulos residuales en `empresa` (fracción despreciable, sin patrón identificado).
7. Se guardó el dataset limpio, listo para la etapa de modelado.

**Pendiente para el siguiente notebook (modelado):** decidir la codificación de las categóricas (incluyendo qué hacer con `ruta_origen`/`empresa`), separar train/test, entrenar y evaluar el modelo.
